# 02. LoRA + SFT 학습 — 이커머스 고객 지원 도메인 특화


> **시나리오 안내** — 이 실습은 가상의 이커머스 마켓플레이스 **쇼핑몰**(자체 PB 라인: 베이직 · 프리미엄 · 라이트 · 한정판)를 소재로 합니다.

| 항목 | 내용 |
|------|------|
| 목적 | 01에서 만든 합성 데이터로 Qwen3.5-2B를 이커머스 고객 상담원으로 변환 |
| Base 모델 | Qwen3.5-2B (2026-02 공개) |
| 학습 방식 | LoRA (Unsloth, FP16) |
| 런타임 | T4 GPU (Colab 무료) |
| 소요 시간 | ~35~40분 |
| 예상 비용 | $0 (Colab 무료 + HF Hub 무료) |

**이 노트북에서 만드는 것:**
```
MyDrive/product-cs-sft-lab/sft-adapter/         # LoRA 어댑터 (~41.9MB, Google Drive)
https://huggingface.co/<user>/product-cs-...    # FP16 merged 모델 (~4GB, HF Hub)
```

> **사전 조건**: 01 노트북에서 `train.jsonl` / `test.jsonl`이 이미 생성되어 있어야 합니다.

---
## Phase 1: 환경 설정

In [5]:
# 필요 패키지 설치
# trl 버전 핀을 제거 — 최신 transformers(processing_class API)와 호환되는 최신 trl 사용
!pip install --quiet unsloth
!pip install --quiet --no-deps "trl>=0.12" peft accelerate bitsandbytes
print("설치 완료 ✅")

In [6]:
import torch
from unsloth import FastLanguageModel

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Phase 2: Base 모델 로드 (Qwen3.5-2B)

HF Hub에서 자동 다운로드 → GPU에 FP16으로 로드.

- `load_in_4bit=False`: FP16 그대로 유지 (04 양자화 실습에서 극적 Before/After 보이기 위해)
- 첫 실행 시 ~4GB 다운로드 (1~3분 소요)
- 같은 세션 재실행 시 로컬 캐시에서 즉시 로드

### 💡 VRAM 사용량은 어떻게 변하나?

지금은 **모델 로드만** 했으니 VRAM 사용량이 ~4GB 근처일 겁니다. Phase 5에서 학습이 시작되면 **약 1GB 정도 늘어 5GB 안팎**이 돼요. 학습은 모델 weight 외에 세 가지를 추가로 메모리에 들고 있어야 하기 때문입니다:

- **Activations (중간 계산값)** — 학생이 시험 문제를 풀 때 **풀이 과정을 종이에 적어두는 것**과 같습니다. 모델이 답을 내기까지의 레이어별 중간 결과를 임시 보관해야 backward pass에서 기울기를 계산할 수 있어요.
- **Gradient (수정 방향)** — 답이 틀렸을 때 **"어느 쪽으로 얼마나 고쳐야 정답에 가까워지는지"** 알려주는 화살표.
- **Optimizer state (학습 기록)** — **운동 코치가 "어제 훈련 어땠는지" 노트에 적어두는 것**처럼, 이전 단계의 momentum과 variance를 기억해 다음 업데이트를 안정적으로 만들어 줍니다.

일반적인 학습이라면 6~10GB까지 부풀 수 있는데, 이 노트북은 **세 가지 최적화로 ~1GB 증가에 그치게** 만들었어요:

- **`use_gradient_checkpointing="unsloth"`** (Phase 4) — Activations를 일부 재계산해서 30~40% 절약 (가장 큰 비중).
- **`optim="adamw_8bit"`** (Phase 5) — Optimizer state를 8비트로 압축 (일반 AdamW 대비 ~75% 절약).
- **LoRA (0.5%만 학습)** — Gradient/Optimizer state가 전체 20억 파라미터가 아니라 **~10M개 LoRA 어댑터에만** 붙음.

이 세 옵션 다 끄면 10~12GB까지 가서 T4 한계(15.8GB)에 거의 닿아요. **5GB 안팎으로 끝나는 건 Unsloth + LoRA + 8bit AdamW 조합의 핵심 가치 증명**입니다.


In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-2B",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=False,
)

used_gb = torch.cuda.memory_allocated() / 1e9
print(f"✅ 모델 로드 완료")
print(f"VRAM 사용량: {used_gb:.2f} GB (T4 16GB 중)")

### `from_pretrained` 내부 동작 흐름

한 줄의 호출이 실행되는 동안 내부에서 이런 순서로 처리됩니다.

```
FastLanguageModel.from_pretrained("Qwen/Qwen3.5-2B", dtype=torch.float16, ...)
                        │
                        ▼
        ┌────────────────────────────────────┐
        │  1. URL 조합                          │
        │     "Qwen/Qwen3.5-2B"                │
        │       → huggingface.co/Qwen/Qwen3.5-2B│
        └─────────────────┬──────────────────┘
                          ▼
        ┌────────────────────────────────────┐
        │  2. 로컬 캐시 조회                      │
        │     ~/.cache/huggingface/hub/         │
        └─────────────────┬──────────────────┘
                          │
                ┌─── 캐시에 있음? ───┐
                │                   │
              YES                  NO
                │                   │
                │                   ▼
                │   ┌──────────────────────────────┐
                │   │  3. HTTPS 다운로드              │
                │   │     config.json                │
                │   │     model-*.safetensors        │
                │   │     model.safetensors.index.json│
                │   │     tokenizer.json             │
                │   │     tokenizer_config.json …    │
                │   │     (~4GB, Colab 기준 1~3분)     │
                │   └─────────────┬──────────────┘
                │                 │
                └────────┬────────┘
                         ▼
        ┌────────────────────────────────────┐
        │  4. 파일 파싱                         │
        │     config.json → 모델 구조 복원        │
        │     .safetensors → PyTorch 텐서 로드   │
        └─────────────────┬──────────────────┘
                          ▼
        ┌────────────────────────────────────┐
        │  5. GPU로 이동                        │
        │     FP16 변환 + CUDA 메모리 업로드       │
        └─────────────────┬──────────────────┘
                          ▼
        ┌────────────────────────────────────┐
        │  6. Unsloth 최적화 패치                │
        │     기본 레이어 → Triton 커널 버전으로 교체 │
        └─────────────────┬──────────────────┘
                          ▼
        ┌────────────────────────────────────┐
        │  7. 토크나이저도 같은 방식으로 로드       │
        └─────────────────┬──────────────────┘
                          ▼
                return (model, tokenizer)
```

**핵심 포인트**:
- 브라우저에서 repo를 찾아 파일을 수동으로 다운로드하고 PyTorch로 로드하는 **모든 과정**이 한 줄에 숨겨져 있음
- 같은 세션에서 재실행하면 **2번에서 캐시 확인 후 3번을 건너뜀** (인터넷 사용 안 함, 수초 내 완료)
- 6번 Unsloth 패치 덕분에 Colab T4 같은 저사양 GPU에서도 LoRA 학습이 돌아감

---
## Phase 3: 학습 데이터 로드

01 실습에서 만든 `train.jsonl`을 로드하고 Unsloth SFT 포맷으로 변환합니다.

In [8]:
import json
from pathlib import Path

# 로컬 경로 우선, 없으면 Google Drive에서 로드 (01 노트북이 Drive에 저장)
LOCAL_PATH = Path("product-cs-sft-lab/sft-dataset/train.jsonl")
if LOCAL_PATH.exists():
    TRAIN_PATH = LOCAL_PATH
    print(f"📂 로컬 경로 사용: {TRAIN_PATH}")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    TRAIN_PATH = Path("/content/drive/MyDrive/product-cs-sft-lab/sft-dataset/train.jsonl")
    print(f"☁️  Drive 경로 사용: {TRAIN_PATH}")

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"train.jsonl을 찾을 수 없습니다.\n"
        f"01 노트북의 Phase 6(Google Drive 저장)을 먼저 실행했는지 확인하세요."
    )

train_data = []
with open(TRAIN_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            train_data.append(json.loads(line))

print(f"✅ 학습 데이터 로드: {len(train_data)}개")
print(f"첫 샘플 구조: {list(train_data[0].keys())}")

### 잠깐, 토크나이저(Tokenizer)가 뭔가요?

모델은 사실 사람이 쓴 텍스트를 **직접 읽지 못합니다.** 내부적으로는 **숫자만** 처리합니다. 그래서 **텍스트 ↔ 숫자 변환기**가 필요한데, 이게 **토크나이저**입니다.

- 텍스트를 **토큰(token)**이라는 작은 조각으로 쪼갠 뒤 각 토큰을 고유한 숫자 ID로 매핑합니다.
- 반대로 모델이 숫자를 출력하면 텍스트로 다시 조립합니다.
- 모델마다 **자기만의 토크나이저**를 씁니다. 그래서 `model`과 `tokenizer`를 항상 **세트로** 로드해야 합니다 (위 **Phase 2 — Base 모델 로드** 섹션에서 `model, tokenizer = FastLanguageModel.from_pretrained(...)`로 둘을 동시에 받은 이유).

아래 셀에서 **Qwen3.5-2B의 토크나이저가 실제로 어떻게 동작하는지** 눈으로 확인해봅시다.

In [9]:
# 토크나이저 동작 확인
# Qwen3.5-2B는 멀티모달(Vision-Language) 모델이라
# from_pretrained가 반환한 "tokenizer"가 실제로는 Qwen3VLProcessor.
# → 내부 텍스트 토크나이저에 접근하려면 .tokenizer 속성을 사용.
tok = getattr(tokenizer, "tokenizer", tokenizer)

sample_text = "밤에 사진 찍으면 색이 이상해요"

# 1) 텍스트 → 토큰 ID
token_ids = tok.encode(sample_text, add_special_tokens=False)
print(f"입력 텍스트:   {sample_text}")
print(f"토큰 ID:      {token_ids}")
print(f"토큰 개수:    {len(token_ids)}개")

# 2) 각 토큰 ID가 어떤 문자열 조각인지 확인
tokens = [tok.decode([tid]) for tid in token_ids]
print(f"\n개별 토큰:    {tokens}")

# 3) 토큰 ID를 다시 텍스트로 복원
restored = tok.decode(token_ids)
print(f"\n복원된 텍스트: {restored}")


In [10]:
from datasets import Dataset

# Qwen3.5 Processor 대신 내부 텍스트 tokenizer 사용 (일관성)
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

def to_sft_text(example):
    """messages 배열을 tokenizer의 chat template으로 변환"""
    return {
        "text": text_tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = Dataset.from_list(train_data)
dataset = dataset.map(to_sft_text)

print(f"변환 완료: {len(dataset)}개")
print("\n첫 샘플(변환 후):")
print("-" * 60)
print(dataset[0]["text"][:500])
print("-" * 60)

---
## Phase 4: LoRA 설정

11강의 **"교과서에 포스트잇 붙이기"**를 코드로 구현합니다.  
전체 20억 개 가중치는 그대로 두고, 옆에 작은 어댑터만 학습합니다 (전체의 ~0.5%).

In [11]:
# LoRA 어댑터 설정 — 11강 "교과서에 포스트잇 붙이기"를 코드로 구현
# 원본 모델(20억 가중치)은 그대로 두고, 아래 설정대로 "포스트잇"만 붙여 학습한다.

from peft import PeftModel

# 이미 LoRA가 붙어있으면 건너뛴다 (셀 재실행 안전)
if isinstance(model, PeftModel):
    print("⚠️  LoRA 어댑터가 이미 붙어있습니다 (이 셀을 이전에 실행함).")
    print("   → 새 값으로 다시 설정하려면 Phase 2-2 (모델 로드)부터 다시 실행하세요.")
else:
    model = FastLanguageModel.get_peft_model(
        model,

        # 포스트잇 한 장의 크기 (rank).
        # 크게(16, 32, 64): 표현력↑, 메모리↑, overfitting 위험↑
        # 작게(4): 메모리 절약·속도↑, 복잡한 지식 못 담을 수 있음
        # 8은 실무 표준 출발점.
        r=8,

        # 포스트잇을 어느 레이어에 붙일지.
        # q_proj, k_proj, v_proj, o_proj → Attention 레이어
        # gate_proj, up_proj, down_proj  → MLP 레이어
        # 전부 지정하면 품질 최고, 일부만 지정하면 메모리 절약.
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],

        # 포스트잇에 적은 메모를 본문에 "얼마나 크게 반영할지" 음량 조절기.
        # 실질 학습 신호 = alpha/r = 16/8 = 2배 증폭.
        # 작으면 LoRA가 거의 안 먹고, 크면 과하게 휘둘림. 관례적으로 r × 2가 무난.
        lora_alpha=16,

        # 학습 중에 어댑터 일부 뉴런을 임의로 가려서 overfitting(데이터를 통째로
        # 외워버리는 현상)을 막는 장치. 시험 공부할 때 일부 페이지를 일부러 가리고
        # 외우는 것과 비슷하다.
        # 소규모 데이터(280개)는 가뜩이나 외울 게 적어서 가리면 학습 부족 → 0.
        # 대규모(수만 개)는 모델이 다 외우려 들어서 0.05~0.1로 일부러 가려준다.
        lora_dropout=0,

        # bias 파라미터(각 뉴런의 절편값)는 학습 대상에서 제외.
        # 왜 LoRA에선 빼나?
        #  (1) LoRA 철학은 "최소 파라미터로 최대 효과" — bias는 weight 행렬에 비해
        #      학습 신호 기여가 미미해서 굳이 학습 대상에 넣을 이유가 없다.
        #  (2) LoRA는 weight 행렬에 저차원 분해(B·A)를 더하는 기법인데
        #      bias는 행렬이 아니라 벡터라 이 분해 구조에 맞지 않는다.
        #  (3) 원 LoRA 논문에서도 "bias 학습해도 품질 차이 거의 없음"이 확인되어
        #      학계·실무 표준으로 굳어졌다.
        # 옵션값: "none"=bias 전부 동결 / "all"=전부 학습 / "lora_only"=LoRA 대상 레이어 bias만
        bias="none",

        # Gradient Checkpointing: 중간 결과를 저장하지 않고 역전파 때 재계산.
        # "unsloth" → Unsloth 전용 최적화 버전. 메모리 30~40% 절약, 속도 약 20% 느려짐.
        # T4 같은 소형 GPU에서는 사실상 필수.
        use_gradient_checkpointing="unsloth",

        # 재현성을 위한 랜덤 시드. 같은 seed면 LoRA 초기값이 항상 동일.
        random_state=42,
    )
    print("✅ LoRA 어댑터 추가 완료")

# 학습 대상 파라미터 비율 확인 — LoRA의 핵심 지표
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\n학습 대상 파라미터: {trainable:,}개")
print(f"전체 파라미터:     {total:,}개")
print(f"비율:            {trainable/total*100:.4f}%")
# ★ 전체의 0.1~1% 수준만 학습된다는 게 확인되어야 정상 (포스트잇 방식이 작동 중)

---
## Phase 5: LoRA + SFT 학습 실행

- **num_epochs=2**: 296개를 2번 반복 (1~3 적정 범위)
- **learning_rate=2e-4**: 적당한 "보폭"
- **batch_size=4 + gradient_accumulation=4**: 유효 batch 16으로 안정 학습
- 예상 소요: **~3~5분** (Unsloth + LoRA + T4 조합이라 빠름)

loss 숫자가 내려가는지 눈으로 확인하세요.

In [12]:
from trl import SFTTrainer, SFTConfig

# Qwen3.5는 멀티모달(Vision-Language) 모델이라
# from_pretrained가 반환한 `tokenizer`는 사실 Qwen3VLProcessor.
# SFTTrainer는 순수 텍스트 tokenizer를 기대하므로 내부 .tokenizer만 꺼내서 넘긴다.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# 최신 trl(>=0.12)은 SFTConfig로 하이퍼파라미터를 넘긴다.
# (구버전에서는 TrainingArguments + tokenizer= 를 썼는데, 새 transformers와 호환성 문제로 변경)
sft_config = SFTConfig(
    # --- 1. 핵심 파라미터 ---
    num_train_epochs=2,                 # 전체 데이터셋을 몇 바퀴 돌지. 1~3 권장. 크면 과적합, 작으면 덜 배움.
    # 한 스텝당 가중치를 얼마나 움직일지의 "보폭".
    # LoRA는 Full fine-tuning(1e-5)보다 100배 큰 보폭(2e-4)을 쓴다. 왜?
    #  (1) Base 모델이 동결돼 있어 큰 보폭이 원본 지식을 망칠 위험이 없음.
    #      (full fine-tuning은 큰 보폭이면 사전학습 지식이 무너짐)
    #  (2) LoRA 어댑터의 B 행렬은 0으로 초기화돼 시작 시 신호가 0 →
    #      작은 보폭이면 의미 있는 값까지 도달이 너무 느림.
    #  (3) 학습 대상이 0.5%(어댑터)뿐이라 영향 범위가 좁음 → 큰 보폭이 안전.
    learning_rate=2e-4,
    per_device_train_batch_size=4,      # GPU 1장당 한 번에 처리할 샘플 수. 크면 안정적이지만 OOM 위험. T4는 4가 안정적.
    gradient_accumulation_steps=4,      # N번 모아서 1번 업데이트. 실제 batch=4 × 누적=4 → 유효 batch 16 효과.

    # --- 2. 학습률 관련 ---
    warmup_steps=10,                    # 처음 N step은 lr을 0부터 서서히 올림. 초반 발산 방지용 (전체 38 step 중 10 step).
    lr_scheduler_type="linear",         # lr 감소 방식. linear=직선 하강. 다른 옵션: cosine(곡선), constant(고정).

    # --- 3. 메모리/속도 최적화 ---
    optim="adamw_8bit",                 # Optimizer 종류. 8bit AdamW는 일반 AdamW 대비 메모리 ~75% 절약.
    fp16=True,                          # 16bit 부동소수로 학습. T4에서 속도·메모리 이득. A100은 bf16 권장.

    # --- 4. 출력, 재현성, SFT 전용 ---
    output_dir="outputs",               # 체크포인트·로그 저장 폴더. 재학습하면 덮어쓴다.
    logging_steps=10,                   # 몇 step마다 loss를 찍을지. 작게 하면 촘촘, 크게 하면 드문드문.
    seed=42,                            # 난수 시드. 같은 seed + 같은 데이터 = 같은 결과 (디버깅/비교 필수).
    dataset_text_field="text",          # 학습에 쓸 컬럼명. Phase 3에서 만든 "text" 필드를 학습 대상으로 지정.
    max_seq_length=2048,                # 샘플 하나의 최대 토큰 수. 초과분은 잘림. 크면 메모리↑, 작으면 긴 대화 잘림.
    packing=False,                      # 짧은 샘플 여러 개를 이어붙여 한 번에 처리하는 최적화. 학습 안정성 위해 OFF.
)

trainer = SFTTrainer(
    model=model,
    processing_class=text_tokenizer,   # 최신 trl API (기존 tokenizer= 파라미터 대체)
    train_dataset=dataset,
    args=sft_config,
)

trainer_stats = trainer.train()

---
## Phase 5-1: Loss 그래프로 학습 추이 확인

학습이 끝났으면 **loss가 정말 내려갔는지**를 숫자 로그(`Step 10: 2.32, Step 20: 1.72, ...`)만 보지 말고 **그래프로** 확인합니다.

- **정상 패턴**: 초반에 빠르게 떨어지다가 후반에 완만해지는 L자 곡선
- **이상 패턴**:
  - **수평선** → learning_rate가 너무 작거나 데이터가 이상함
  - **오락가락 튐** → batch_size가 너무 작음
  - **발산(증가)** → learning_rate가 너무 큼


In [13]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import subprocess
from pathlib import Path

# --- Colab 한글 폰트 (한 번만 설치) ---
nanum_path = Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
if not nanum_path.exists():
    try:
        subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"],
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        pass
if nanum_path.exists():
    fm.fontManager.addfont(str(nanum_path))
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

# --- trainer.state.log_history에서 loss 추출 ---
# log_history는 {step, loss, epoch, learning_rate, ...} dict의 리스트
log_history = trainer.state.log_history
steps  = [e["step"] for e in log_history if "loss" in e]
losses = [e["loss"] for e in log_history if "loss" in e]

if not losses:
    print("⚠️ log_history에 loss가 없습니다. trainer_stats를 직접 확인하세요:")
    print(trainer_stats)
else:
    print(f"총 {len(losses)}개 로그 포인트")
    print(f"첫 loss: {losses[0]:.4f} (step {steps[0]})")
    print(f"끝 loss: {losses[-1]:.4f} (step {steps[-1]})")
    print(f"감소율:  {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")

    # --- 그래프 ---
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps, losses, marker="o", linewidth=2.5, markersize=8, color="#3b82f6")
    ax.set_xlabel("Training Step", fontsize=12)
    ax.set_ylabel("Training Loss", fontsize=12)
    ax.set_title("SFT 학습 Loss 추이 — 아래로 내려가는 곡선이 정상", fontsize=13)
    ax.grid(True, alpha=0.3, linestyle="--")

    # 첫·끝 값 주석
    ax.annotate(f"{losses[0]:.3f}", (steps[0], losses[0]),
                textcoords="offset points", xytext=(0, 12), ha="center", fontsize=10, color="#ef4444")
    ax.annotate(f"{losses[-1]:.3f}", (steps[-1], losses[-1]),
                textcoords="offset points", xytext=(0, -18), ha="center", fontsize=10, color="#10b981")

    plt.tight_layout()
    plt.show()

    # --- 간이 진단 ---
    print("\n💡 진단:")
    drop_pct = (losses[0] - losses[-1]) / losses[0] * 100
    if drop_pct > 20:
        print(f"   ✅ 정상: loss가 {drop_pct:.0f}% 감소 — 모델이 학습 신호를 잘 받았음")
    elif drop_pct > 0:
        print(f"   ⚠️ 약함: loss가 {drop_pct:.0f}%만 감소 — epoch 늘리거나 lr 조정 고려")
    else:
        print(f"   ❌ 비정상: loss가 오히려 증가 — lr이 너무 크거나 데이터 문제 가능성")


---
## Phase 6: 학습 전/후 답변 확인 (눈 체감)

본격적인 평가는 03 실습에서. 여기서는 답변 톤이 실제로 바뀌었는지 빠르게 확인합니다.

In [14]:
# 학습 모드 → 추론 모드 전환 (gradient 계산 메모리 해제, KV cache ON → 속도 ~2배)
FastLanguageModel.for_inference(model)

# Qwen3.5는 멀티모달 모델이라 from_pretrained가 돌려준 tokenizer는 Processor 덩어리.
# 내부에 숨은 순수 텍스트 토크나이저를 꺼내서 써야 apply_chat_template / decode가 안전.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# 테스트 질문 — train.jsonl에 정확히 같은 문장은 없고 "야간 촬영" 주제만 겹침.
# 학습 문장을 그대로 외웠는지가 아니라, 상담원 톤으로 일반화됐는지를 본다.
test_question = "밤에 사진 찍으면 색이 이상해요"

# LLM은 [{role, content}] 리스트 형태로 대화를 받음.
# 지금은 사용자가 질문 하나 던지는 상황이라 user 항목만 1개.
messages = [{"role": "user", "content": test_question}]

# messages를 모델이 알아듣는 형식(<|im_start|>user ... <|im_end|>)으로 변환.
inputs = text_tokenizer.apply_chat_template(
    messages,
    tokenize=True,              # 문자열까지만이 아니라 토큰 ID 숫자까지 변환
    add_generation_prompt=True, # 끝에 "<|im_start|>assistant" 붙여서 "이제 답할 차례" 신호
    return_tensors="pt",        # PyTorch 텐서(GPU가 계산할 수 있는 숫자 덩어리)로 반환
).to("cuda")                    # 모델이 GPU에 있으므로 입력도 GPU 메모리로 이동

# 실제 답변 생성 — inputs 뒤에 토큰을 한 개씩 이어 붙여나감
outputs = model.generate(
    inputs,
    max_new_tokens=300,   # 최대 300 토큰까지만 생성 (한국어 400~500자 분량). 너무 길면 장황해짐.
    temperature=0.3,      # "창의성 vs 일관성" 다이얼. 0=항상 같은 답, 1=매번 다른 답. 상담은 일관성 우선.
    do_sample=True,       # 확률 기반 샘플링 ON. False면 greedy(최고 확률 하나). temperature가 의미 있으려면 True.
)

# outputs[0]은 [입력 토큰 + 새로 생성된 토큰] 전체가 붙어있음.
# inputs.shape[1]: 로 잘라내서 "새로 나온 부분"만 디코딩 (안 하면 앞에 질문이 되풀이됨).
# skip_special_tokens=True → <|im_start|> 같은 내부용 토큰 숨김
answer = text_tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(f"질문: {test_question}\n")
print(f"답변:\n{answer}")

---
## Phase 7: 저장 및 HF Hub 배포

- **Google Drive 저장**: LoRA 어댑터만 (~41.9MB) — 세션이 끊겨도 남음
- **Hub 배포**: Base + LoRA 합친 FP16 merged 모델 (~4GB) → 03(평가)/04(양자화) 실습에서 재사용

In [15]:
import os
from pathlib import Path

# Processor가 아닌 내부 text tokenizer를 저장 (이후 로드 시 호환성 좋음)
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# 저장 위치 결정 — Colab이면 Google Drive, 아니면 로컬 디스크
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER_DIR = Path("/content/drive/MyDrive/product-cs-sft-lab/sft-adapter")
    print(f"☁️  Google Drive에 저장: {ADAPTER_DIR}")
except ModuleNotFoundError:
    ADAPTER_DIR = Path("sft-adapter")
    print(f"📂 로컬 디스크에 저장 (Colab 아님): {ADAPTER_DIR}")

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(ADAPTER_DIR)) # LoRA 가중치만 저장 (Base 모델은 저장 안됨)
text_tokenizer.save_pretrained(str(ADAPTER_DIR)) # 토크나이저도 저장. 04(양자화 실습)에서 GGUF 변환할 때 같은 토크나이저 필요.

adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
) / 1e6

print(f"\n✅ 어댑터 저장 완료: {ADAPTER_DIR}")
print(f"파일 크기: {adapter_size:.1f} MB")

In [16]:
import os
from getpass import getpass
from huggingface_hub import login

# HF 토큰 발급 방법:
#   1) https://huggingface.co/settings/tokens 접속
#   2) "New token" 클릭 → Type: Write 선택 (업로드에는 write 권한 필수, read로는 불가)
#   3) 생성된 토큰을 복사해서 아래 입력란에 붙여넣고 Enter
token = getpass("HF_TOKEN: ").strip()
if not token:
    raise RuntimeError("❌ HF 토큰이 입력되지 않았습니다.")

# 환경변수 + huggingface_hub 세션 둘 다 등록 → 이후 셀에서 token=True로 자동 사용
os.environ["HF_TOKEN"] = token
login(token=token, add_to_git_credential=False)
print("✅ HF 인증 완료")
print("인증 에러나면 https://huggingface.co/settings/tokens 에서 Type: Write 토큰 다시 발급 필요")

In [17]:
# (옵션) 기존 repo가 LFS 포인터 오류로 깨진 경우 초기화.
# 이전 push가 중단되어 "tokenizer.json 파일이 없다"는 BadRequestError가 나면 이 셀을 실행.
# 아직 repo가 없거나(첫 실행) 정상 상태면 건너뛰어도 된다.
from huggingface_hub import whoami, delete_repo
from huggingface_hub.errors import RepositoryNotFoundError

HF_USERNAME = whoami()["name"]
REPO_NAME = f"{HF_USERNAME}/product-cs-qwen3.5-2b-sft"

try:
    delete_repo(repo_id=REPO_NAME, repo_type="model", token=True)
    print(f"🗑️  기존 repo 삭제 완료: {REPO_NAME}")
except RepositoryNotFoundError:
    print(f"ℹ️  삭제할 repo 없음 (첫 업로드): {REPO_NAME}")

In [18]:
# whoami()로 현재 로그인된 HF 계정을 자동 감지 (Phase 7-2의 login() 완료 전제)
from huggingface_hub import whoami

try:
    HF_USERNAME = whoami()["name"]
except Exception as e:
    raise RuntimeError(
        "❌ HF 로그인이 완료되지 않았습니다.\n"
        "   Phase 7-2 셀에서 HF 토큰을 입력해 '✅ HF 인증 완료' 메시지가 뜬 뒤\n"
        "   이 셀을 다시 실행하세요.\n"
        "   (HF 토큰 발급: https://huggingface.co/settings/tokens — Write 권한)"
    ) from e

REPO_NAME = f"{HF_USERNAME}/product-cs-qwen3.5-2b-sft"
print(f"HF 계정: {HF_USERNAME}")
print(f"업로드 대상 repo: {REPO_NAME} (private)")

# Processor 대신 text tokenizer를 함께 업로드 (멀티모달 래퍼 없이 순수 LLM으로 사용 가능하게)
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

model.push_to_hub_merged(
    REPO_NAME,
    text_tokenizer,
    save_method="merged_16bit",
    token=True,
    private=True,   # 기본 private으로 업로드 — 필요 시 HF 웹 Settings에서 공개 전환 가능
)

print(f"✅ Hub 배포 완료: https://huggingface.co/{REPO_NAME}")

---
## 완료

**전체 산출물:**

- **LoRA 어댑터** → `MyDrive/product-cs-sft-lab/sft-adapter/` (~41.9MB)
- **FP16 merged 모델** → `https://huggingface.co/<user>/product-cs-qwen3.5-2b-sft` (~4GB)
- **학습 로그** → 이 노트북 셀 출력 (loss 감소 추이)

**다음 실습:**
- `03_sft_eval.ipynb` — 학습 전/후 4조건 평가 + Langfuse 연동
- `04_quantization.ipynb` — FP16 → GGUF 4bit 양자화 (극적 크기 축소)

> ⚠️ **test.jsonl은 이번 학습 노트북에 절대 입력하지 않았습니다.** 03 평가에서만 사용됩니다.